# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. All entities are referenced by their `@id` fields as defined in the metadata.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The `@id` fields provide stable references for each entity. Let's inspect what record sets are present and their fields.

Note: The Croissant schema organizes tabular and related data into Record Sets (datasets, tables, files, etc.).

In [ ]:
# List all available record set `@id`s
print("Available Record Sets (by @id):")
for rs in metadata.record_sets:
    print(f"- {rs['@id']}")

# As a demonstration, show fields for the main record set
# (Find the primary record set with tabular data)
main_record_set = None
for rs in metadata.record_sets:
    # Heuristically, select the record set with columns
    if rs.get('fields', None):
        main_record_set = rs
        break

if main_record_set:
    print(f"\nMain Record Set: {main_record_set['@id']}")
    print("Fields / Columns in this record set:")
    for fld in main_record_set['fields']:
        print(f"- {fld['@id']} (name: {fld['name']}, type: {fld.get('dataType', 'N/A')})")
else:
    print("No main record set found.")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All record and field references use their `@id` values for clarity and reproducibility.

In [ ]:
# Select all record sets with tabular data
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record_set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
    else:
        print(f"No tabular records found for {record_set_id}.")

# Use the main record set as identified previously
main_id = main_record_set['@id'] if main_record_set is not None else record_set_ids[0]
if main_id in dataframes:
    print(f"\nShowing first 5 rows from main record set {main_id}:")
    display(dataframes[main_id].head())
else:
    print(f"No data for main record set: {main_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records on a numeric field, normalizing, and grouping. For this example, we'll work with one of the main numeric fields (e.g., "MW" for molecular weight, or similar as present in the schema). All fields are referenced by their `@id`.

Let's list the available fields and pick a numeric one for demonstration:

In [ ]:
df = dataframes[main_id]

all_fields = [fld for fld in main_record_set['fields']]
print("Fields in main record set with @id and type:")
for fld in all_fields:
    print(f"- {fld['@id']}: {fld['name']}, type={fld.get('dataType')}")

# Let's try to use the 'MW' (molecular weight), often a Float or Integer, as an example
numeric_field_id = None
for fld in all_fields:
    if fld['name'].lower() in ['mw', 'molecular_weight', 'molecularweight', 'molecular weight'] or fld.get('dataType') in ['Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer']:
        numeric_field_id = fld['@id']
        break

if not numeric_field_id:
    # fallback: select a numeric-looking field
    for fld in all_fields:
        candidate = fld['name']
        if candidate.lower() not in ['accession', 'name', 'description']:
            numeric_field_id = fld['@id']
            break

print(f"\nUsing numeric field '@id': {numeric_field_id}")
numeric_column = None
for col in df.columns:
    if numeric_field_id in col or col.lower() in ['mw', 'molecular_weight', 'molecular weight']:
        numeric_column = col
        break
if not numeric_column:
    numeric_column = df.select_dtypes(include='number').columns[0]

print(f"Underlying column: {numeric_column}")

# Filtering example
threshold = df[numeric_column].mean() if pd.api.types.is_numeric_dtype(df[numeric_column]) else 10
filtered_df = df[df[numeric_column] > threshold]
print(f"Filtered records with {numeric_column} > {threshold}:")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_column}_normalized"] = (
    filtered_df[numeric_column] - filtered_df[numeric_column].mean()
) / (filtered_df[numeric_column].std() + 1e-8)
print(f"Normalized values for filtered {numeric_column}:")
display(filtered_df[[numeric_column, f"{numeric_column}_normalized"]].head())

# Try grouping by a text/categorical field if available
group_field_id = None
for fld in all_fields:
    if fld['@id'] != numeric_field_id and fld.get('dataType', '').lower() in ['text', 'string']:
        group_field_id = fld['@id']
        break
if not group_field_id:
    for fld in all_fields:
        if fld['@id'] != numeric_field_id:
            group_field_id = fld['@id']
            break

group_column = None
for col in df.columns:
    if group_field_id and group_field_id in col:
        group_column = col
        break
if not group_column:
    # fallback, pick a string column
    group_candidates = df.select_dtypes(include=['object']).columns
    if len(group_candidates) > 0:
        group_column = group_candidates[0]

if group_column:
    grouped_df = filtered_df.groupby(group_column)[numeric_column].mean().reset_index()
    print(f"Grouped mean values by '{group_column}':")
    display(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, we plot the distribution of the chosen numeric field and, if grouping data is available, show grouped averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x=numeric_column, kde=True)
plt.title(f'Distribution of {numeric_column}')
plt.xlabel(numeric_column)
plt.ylabel('Count')
plt.show()

# If group_column available, boxplot
if group_column:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_column, y=numeric_column, data=df)
    plt.title(f'{numeric_column} by {group_column}')
    plt.ylabel(numeric_column)
    plt.xlabel(group_column)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we:
- Explored the metadata and structure of the FAIR² mass spectrometry dataset using the `mlcroissant` library
- Referenced all entities by their stable `@id` as specified in the schema
- Loaded records from the primary record set into pandas
- Performed basic EDA: field inspection, numeric filtering/normalization, and grouped analysis
- Visualized the distribution of a main numeric measurement and grouped view by a categorical/text field

This workflow can be extended to more complex analysis and to interoperate with other FAIR datasets using the Croissant standard.